# Phishing Email Detection — Exploration

Quick EDA: class balance, text lengths, URL signals, top TF-IDF terms.
Run from project root. Static analysis only (never opens URLs).

In [ ]:
import pandas as pd
from src.data_loader import load_dataset, print_statistics
from src.preprocessing import clean_dataframe
from src.url_features import url_features_to_matrix, FEATURE_NAMES

raw = load_dataset('data/raw/emails.csv')
print_statistics(raw)
df = clean_dataframe(raw)
print(df.head(3).to_string())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df['label'].value_counts().plot(kind='bar')
plt.xticks([0, 1], ['Safe', 'Phishing'])
plt.title('Class distribution')
plt.show()

df['length'] = df['combined_text'].str.len()
sns.histplot(data=df, x='length', hue='label', bins=30)
plt.title('Email length by class')
plt.show()

In [ ]:
mat = url_features_to_matrix(df['combined_text'])
url_df = pd.DataFrame(mat, columns=FEATURE_NAMES)
url_df['label'] = df['label'].values
print(url_df.groupby('label').mean().T.to_string())

In [ ]:
# Top TF-IDF terms per class (EXPLORATION only — training fits on train split)
from src.text_features import build_tfidf_vectorizer, SUSPICIOUS_KEYWORDS
import numpy as np

vec = build_tfidf_vectorizer(max_features=2000)
X = vec.fit_transform(df['combined_text'])
terms = np.array(vec.get_feature_names_out())
for label, name in [(1, 'Phishing'), (0, 'Safe')]:
    idx = df['label'].values == label
    scores = X[idx].mean(axis=0).A1
    print(name, list(terms[scores.argsort()[-15:][::-1]]))
print('Tracked keywords:', SUSPICIOUS_KEYWORDS)